In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class DatasetPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        target_column=None,
        drop_unnamed=True,
        drop_na=True,
        drop_leakage=True,
        drop_constant_features=True
    ):
        self.target_column = target_column
        self.drop_unnamed = drop_unnamed
        self.drop_na = drop_na
        self.drop_leakage = drop_leakage
        self.drop_constant_features = drop_constant_features

    def _get_leakage_columns(self):
        leakage_map = {
            'IC50, mM': ['SI'],
            'CC50, mM': ['SI'],
            'SI': ['IC50, mM', 'CC50, mM']
        }
        return leakage_map.get(self.target_column, [])

    def fit(self, X, y=None):
        X_fit = X.copy()

        # удалить служебный столбец
        if self.drop_unnamed:
            X_fit = X_fit.drop(columns=['Unnamed: 0'], errors='ignore')

        # удалить leakage-колонки
        if self.drop_leakage:
            X_fit = X_fit.drop(columns=self._get_leakage_columns(), errors='ignore')

        # удалить строки с пропусками
        if self.drop_na:
            X_fit = X_fit.dropna()

        # найти и сохранить константные признаки
        if self.drop_constant_features:
            nunique = X_fit.nunique()
            self.constant_features_ = nunique[nunique == 1].index.tolist()
        else:
            self.constant_features_ = []

        return self

    def transform(self, X):
        X = X.copy()

        # удалить служебный столбец
        if self.drop_unnamed:
            X = X.drop(columns=['Unnamed: 0'], errors='ignore')

        # удалить leakage-колонки
        if self.drop_leakage:
            X = X.drop(columns=self._get_leakage_columns(), errors='ignore')

        # удалить строки с пропусками
        if self.drop_na:
            X = X.dropna()

        # удалить константные признаки, найденные на этапе fit
        if self.drop_constant_features:
            X = X.drop(columns=self.constant_features_, errors='ignore')

        return X


# from sklearn.pipeline import Pipeline
#
# pipeline_preprocessing = Pipeline([
#     ('drop_columns', Preprocessor(target_column='IC50, mM'))
# ])